In [2]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import importlib
import utils_bedpe

In [3]:
importlib.reload(utils_bedpe)
from utils_bedpe import *

In [3]:
ct2 = 'BRCA_C9C8D426_A3FD_4455_89A9_768BC01D66A9_X009_S02_B1_T1'
#BRCA_8D1E6006_85CB_484A_8B5C_30766D90137B_X001_S01_B1_T1
data = np.load(f"/data1/lesliec/carolw/projects/chromafold/predictions/BRCA-chan_1_3k/{ct2}/prediction_{ct2}_chr17.npz", allow_pickle=True)
print(data.files)          # list available arrays
mat = data['arr_0']        
print(mat.shape)
print(mat[:5])


['arr_0']
(7819, 400)
[[-0.32188836 -0.2900983  -0.29064164 ... -0.14622563 -0.09975682
  -0.12507562]
 [-0.33363286 -0.30317527 -0.3067571  ... -0.09991793 -0.05387957
  -0.07577198]
 [-0.3835057  -0.3583001  -0.3590613  ...  0.03396162  0.06068178
   0.03445034]
 [-0.42092773 -0.39728367 -0.3950974  ... -0.1998698  -0.15760447
  -0.18191518]
 [-0.4350852  -0.40572816 -0.4035534  ... -0.1800162  -0.14135464
  -0.16478145]]


In [9]:
chrom = 17

cell_types = [
    "BRCA_C147AAD5_A8F1_41D5_8709_21820BE50902_X008_S02_B1_T1"
]

pred_path = f"/data1/lesliec/carolw/projects/chromafold/predictions/BRCA-chan_1_3k"
bedpe_path = f"/data1/lesliec/carolw/projects/chromafold/predictions/BRCA-chan_1_3k"

genome = "hg38"          # whatever get_chrom_starts() expects
loop_size_cutoff = 65_000   # 200 kb
cutoff = 5               

os.makedirs(f"{bedpe_path}/bedpe_files", exist_ok=True)
df=makeBedpe(
    chrom=chrom,
    cell_types=cell_types,
    pred_path=pred_path,
    bedpe_path=bedpe_path,
    cutoff=cutoff,
    genome=genome,
    loop_size_cutoff=loop_size_cutoff,
    percent_co=False
)

In [4]:
# for filtering ground truth to interaction within a specific range
# default 65 bins at 10 kb resolution
ct = "BRCA-7C6A3AE4-E2EA-42B3-B3F1-81C19E6F2170-X004-S01-B1-T1_H3K27ac"
peak_dir= f"/data1/lesliec/carolw/projects/chromafold/preprocess/BRCA-chan/hichip/{ct}"

df = pd.read_csv(f"{peak_dir}/{ct}_hg38_10kb_GATC_GANTC_FDR_05_norm.bedpe", sep="\t", header=0)
# Filter same chrom and within 650kb
max_dist = 65 * 10_000  # 650kb

filtered = df[
    (df["chrom1"] == df["chrom2"]) &
    (df["start2"] - df["start1"] <= max_dist)
]

filtered.to_csv(f"{peak_dir}/{ct}_hg38_10kb_GATC_GANTC_FDR_05_norm_65bins.bedpe", sep="\t", header=True, index=False)